[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/08-time-series.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/08-time-series.ipynb)

# Time Series Analysis: Resampling, Smoothing, Decomposition, and ARIMA

**Module 4 — Data Science & Visualization** | Estimated time: 50 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Build DatetimeIndex-indexed DataFrames and resample to different frequencies
- Compute rolling windows and exponential smoothing
- Decompose a time series into trend, seasonal, and residual components
- Plot ACF and PACF to identify ARIMA parameters
- Fit a simple ARIMA model and generate forecasts
- Generate synthetic time series with controllable trend, seasonality, and noise

In [ ]:
!pip install statsmodels --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 100, 'figure.facecolor': 'white'})

rng = np.random.default_rng(42)
print('Libraries loaded successfully.')

## 1. Building a Synthetic Time Series

We generate a daily retail sales series with:
- A linear **trend** (growing sales)
- A **weekly seasonality** (weekends spike)
- A **yearly seasonality** (holiday boost in December)
- **Random noise**

In [ ]:
# Date range: 3 years of daily data
date_rng = pd.date_range(start='2021-01-01', end='2023-12-31', freq='D')
n = len(date_rng)
t = np.arange(n)

# Components
trend     = 500 + 0.15 * t                                              # gradual growth
weekly    = 80 * np.sin(2 * np.pi * t / 7 + np.pi / 2)                # 7-day cycle
yearly    = 150 * np.sin(2 * np.pi * t / 365.25 - np.pi / 2)           # 365-day cycle
holiday   = np.where((date_rng.month == 12) & (date_rng.day >= 15), 200, 0)
noise     = rng.normal(0, 40, n)

sales     = trend + weekly + yearly + holiday + noise
sales     = np.maximum(sales, 50)  # no negative sales

ts = pd.Series(sales, index=date_rng, name='sales')
ts.index.name = 'date'

print(f'Time series length: {len(ts)} days')
print(f'Date range: {ts.index[0].date()} to {ts.index[-1].date()}')
print(f'Sales stats: min={ts.min():.0f}, max={ts.max():.0f}, mean={ts.mean():.0f}')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(ts.index, ts.values, linewidth=0.8, color='steelblue', alpha=0.8)
ax.set_title('Daily Retail Sales — 3 Years', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Sales ($)')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. DatetimeIndex: pd.to_datetime and Resampling

Pandas DatetimeIndex enables time-aware operations. `.resample()` is the time-series equivalent of `groupby` — it groups data by a time frequency and applies an aggregation.

In [ ]:
# Resampling to different frequencies
ts_weekly  = ts.resample('W').sum()     # weekly total
ts_monthly = ts.resample('ME').sum()    # monthly total (ME = Month End)
ts_quarterly = ts.resample('QE').sum() # quarterly

print('Original (daily):  ', ts.shape)
print('Weekly  resampled: ', ts_weekly.shape)
print('Monthly resampled: ', ts_monthly.shape)
print('Quarterly:         ', ts_quarterly.shape)

# Multiple aggregations at once
ts_monthly_stats = ts.resample('ME').agg(['sum', 'mean', 'std', 'min', 'max'])
print('\nMonthly stats (first 4 months):')
print(ts_monthly_stats.head(4).round(1).to_string())

# Plot comparison
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)

axes[0].plot(ts.index, ts.values, linewidth=0.7, color='steelblue', alpha=0.7)
axes[0].set_title('Daily Sales')
axes[0].set_ylabel('Sales ($)')

axes[1].plot(ts_weekly.index, ts_weekly.values, linewidth=1.5, color='tomato', marker='o', markersize=2)
axes[1].set_title('Weekly Total Sales')
axes[1].set_ylabel('Sales ($)')

axes[2].bar(ts_monthly.index, ts_monthly.values, width=20, color='mediumpurple', alpha=0.8)
axes[2].set_title('Monthly Total Sales')
axes[2].set_ylabel('Sales ($)')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Rolling Windows and Exponential Smoothing

**Rolling mean** smooths noise by averaging over a fixed window. **Exponential Weighted Mean (EWM)** gives more weight to recent observations, making it more responsive to trend changes.

In [ ]:
# Rolling statistics
ts_roll7   = ts.rolling(window=7,   center=True).mean()    # 7-day MA
ts_roll30  = ts.rolling(window=30,  center=True).mean()    # 30-day MA
ts_roll_std= ts.rolling(window=30,  center=True).std()     # 30-day rolling std

# Exponential Weighted Mean
ts_ewm_fast = ts.ewm(span=7,  adjust=False).mean()    # alpha ~ 2/(span+1)
ts_ewm_slow = ts.ewm(span=30, adjust=False).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Rolling means
axes[0].plot(ts.index, ts.values, alpha=0.3, color='steelblue', label='Raw daily', linewidth=0.7)
axes[0].plot(ts_roll7.index,  ts_roll7.values,  color='orangered', linewidth=2, label='7-day MA')
axes[0].plot(ts_roll30.index, ts_roll30.values, color='green',     linewidth=2, label='30-day MA')
# Bollinger-band-style: mean ± 1 std
upper = ts_roll30 + ts_roll_std
lower = ts_roll30 - ts_roll_std
axes[0].fill_between(ts.index, lower, upper, alpha=0.15, color='green', label='30-day ±1 std')
axes[0].set_title('Rolling Window Smoothing')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# EWM
axes[1].plot(ts.index, ts.values,         alpha=0.3, color='steelblue', label='Raw', linewidth=0.7)
axes[1].plot(ts.index, ts_ewm_fast.values, color='tomato', linewidth=2, label='EWM span=7')
axes[1].plot(ts.index, ts_ewm_slow.values, color='purple', linewidth=2, label='EWM span=30')
axes[1].set_title('Exponential Weighted Mean (EWM)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_ylabel('Sales ($)')

plt.tight_layout()
plt.show()

## 4. Seasonal Decomposition

`seasonal_decompose` separates a time series into **trend**, **seasonal**, and **residual** components using a moving average approach. Choose `model='additive'` when seasonal fluctuations are constant or `model='multiplicative'` when they scale with the trend.

In [ ]:
# Use weekly data for cleaner decomposition
result = seasonal_decompose(ts_weekly, model='additive', period=52)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

ts_weekly.plot(ax=axes[0], color='steelblue', linewidth=1.2)
axes[0].set_title('Original Series')
axes[0].set_ylabel('Sales')

result.trend.plot(ax=axes[1], color='tomato', linewidth=2)
axes[1].set_title('Trend Component')
axes[1].set_ylabel('Trend')

result.seasonal.plot(ax=axes[2], color='green', linewidth=1.2)
axes[2].set_title('Seasonal Component (52-week period)')
axes[2].set_ylabel('Seasonal')

result.resid.plot(ax=axes[3], color='gray', linewidth=1, alpha=0.7)
axes[3].axhline(0, color='black', linewidth=0.8)
axes[3].set_title('Residual (noise)')
axes[3].set_ylabel('Residual')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Seasonal Decomposition — Weekly Sales', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Strength of seasonality and trend
var_resid = result.resid.dropna().var()
var_trend  = (result.trend.dropna() + result.resid.dropna()).var()
var_season = (result.seasonal + result.resid.dropna()).var()

print(f'Strength of Seasonality: {max(0, 1 - var_resid / var_season):.3f}')
print(f'Strength of Trend:       {max(0, 1 - var_resid / var_trend):.3f}')

## 5. ACF and PACF Plots

**ACF** (Autocorrelation Function) measures correlation of the series with its own lagged values. **PACF** (Partial ACF) measures the same after removing the effect of shorter lags. Together they help identify ARIMA(p, d, q) parameters:
- Slow ACF decay + PACF cuts off at lag p → AR(p)
- ACF cuts off at lag q + PACF decays → MA(q)

In [ ]:
# Stationarity test (Augmented Dickey-Fuller)
monthly_sales = ts.resample('ME').sum()
adf_result = adfuller(monthly_sales)
print(f'ADF Statistic: {adf_result[0]:.4f}')
print(f'p-value:       {adf_result[1]:.4f}')
print('Stationary:', 'Yes' if adf_result[1] < 0.05 else 'No (consider differencing)')

# Difference to achieve stationarity if needed
monthly_diff = monthly_sales.diff().dropna()
adf_diff = adfuller(monthly_diff)
print(f'\nAfter 1st differencing — p-value: {adf_diff[1]:.4f}')
print('Stationary:', 'Yes' if adf_diff[1] < 0.05 else 'No')

# ACF and PACF
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

plot_acf(monthly_sales, lags=24, ax=axes[0, 0], alpha=0.05)
axes[0, 0].set_title('ACF — Original Monthly Sales')

plot_pacf(monthly_sales, lags=24, ax=axes[0, 1], alpha=0.05, method='ywm')
axes[0, 1].set_title('PACF — Original Monthly Sales')

plot_acf(monthly_diff, lags=24, ax=axes[1, 0], alpha=0.05)
axes[1, 0].set_title('ACF — After 1st Differencing')

plot_pacf(monthly_diff, lags=24, ax=axes[1, 1], alpha=0.05, method='ywm')
axes[1, 1].set_title('PACF — After 1st Differencing')

plt.suptitle('Autocorrelation Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. ARIMA Forecasting

**ARIMA(p, d, q)**: p = autoregressive order, d = differencing degree, q = moving-average order. We fit on the first 30 months and forecast the last 6.

In [ ]:
# Train/test split
train_end = '2023-06-30'
train = monthly_sales[:train_end]
test  = monthly_sales[train_end:]

print(f'Training: {len(train)} months | Test: {len(test)} months')

# Fit ARIMA(2, 1, 2)
model = ARIMA(train, order=(2, 1, 2))
result_arima = model.fit()
print(result_arima.summary().tables[0])

# Forecast
fcast = result_arima.get_forecast(steps=len(test))
fcast_mean = fcast.predicted_mean
fcast_ci   = fcast.conf_int(alpha=0.2)  # 80% CI

# Evaluation
mae  = np.mean(np.abs(test.values - fcast_mean.values))
rmse = np.sqrt(np.mean((test.values - fcast_mean.values) ** 2))
mape = np.mean(np.abs((test.values - fcast_mean.values) / test.values)) * 100
print(f'\nForecast Accuracy (Test Set):')
print(f'  MAE  = {mae:,.0f}')
print(f'  RMSE = {rmse:,.0f}')
print(f'  MAPE = {mape:.1f}%')

# Plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train.index, train.values, color='steelblue', linewidth=2, label='Training data')
ax.plot(test.index,  test.values,  color='green',     linewidth=2, label='Actual (test)')
ax.plot(fcast_mean.index, fcast_mean.values, color='tomato', linewidth=2,
        linestyle='--', label='ARIMA Forecast')
ax.fill_between(fcast_ci.index,
                fcast_ci.iloc[:, 0], fcast_ci.iloc[:, 1],
                alpha=0.25, color='tomato', label='80% CI')
ax.axvline(pd.Timestamp(train_end), color='gray', linestyle=':', linewidth=1.5)
ax.set_title('ARIMA(2,1,2) — Monthly Sales Forecast', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Monthly Sales ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Practical: Multi-product Time Series Comparison

In [ ]:
# Generate 4 products with different growth rates and seasonality strengths
products = {
    'Widget':   {'trend': 0.20, 'season': 100, 'noise': 50},
    'Gadget':   {'trend': 0.05, 'season': 200, 'noise': 80},
    'Gizmo':    {'trend': 0.35, 'season': 50,  'noise': 30},
    'Doohickey':{'trend': -0.05, 'season': 80, 'noise': 60}
}

date_rng_2 = pd.date_range('2022-01-01', '2023-12-31', freq='W')
n2 = len(date_rng_2)
t2 = np.arange(n2)

df_multi = pd.DataFrame(index=date_rng_2)
for name, params in products.items():
    component = (
        300
        + params['trend'] * t2
        + params['season'] * np.sin(2 * np.pi * t2 / 52)
        + rng.normal(0, params['noise'], n2)
    )
    df_multi[name] = np.maximum(component, 10)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Raw series
for col in df_multi.columns:
    axes[0].plot(df_multi.index, df_multi[col], linewidth=1.2, label=col, alpha=0.8)
axes[0].set_title('Weekly Sales — 4 Products')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel('Units')

# 12-week rolling mean comparison
for col in df_multi.columns:
    rm = df_multi[col].rolling(12, center=True).mean()
    axes[1].plot(df_multi.index, rm, linewidth=2, label=f'{col} (12-wk MA)')
axes[1].set_title('12-Week Rolling Average — Trend Comparison')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylabel('Units (smoothed)')

plt.tight_layout()
plt.show()

# Correlation between products
print('Product cross-correlations (weekly):')
print(df_multi.corr().round(3))

## Practice Exercises

**Exercise 1 — Resampling Business Logic**
Using the `ts` series, compute: (a) the best and worst sales month for each year, (b) the week-over-week percentage change in weekly totals, and (c) a quarterly rolling 4-quarter sum (trailing 4 quarters). Plot all three.

**Exercise 2 — EWM vs Rolling Mean**
Generate a step-change series (flat at 100 for 50 periods, then jumps to 200 for 50 periods, with noise std=15). Apply both a 10-period rolling mean and EWM with span=10. Which reacts faster to the step change? Plot the response and explain why.

**Exercise 3 — ARIMA Parameter Selection**
Using the `monthly_sales` series, try three ARIMA configurations: (1,1,1), (2,1,0), and (0,1,2). For each, print the AIC score. Which configuration has the lowest AIC? Does a lower AIC guarantee better forecasting performance on held-out data? Test your answer.